# distill-translate — 02: Early exit (Part 1 + Part 2)

**Goal:** let the t5-small decoder stop after fewer than 6 layers on easy words, and measure
how many layers that saves versus how much translation quality (BLEU) it costs.

**How to use this notebook**
1. Open it from the GitHub link (not a saved copy), so you always get the latest version:
   `https://colab.research.google.com/github/itsani-cmd/Distillation_cost_saver/blob/early-exit-part1/notebooks/02_early_exit.ipynb`
2. **Runtime → Change runtime type → GPU.**
3. Edit only the **Settings** cell, then **Runtime → Run all**.

You never need to edit the other cells: they read the settings and always pull the newest code.
Training resumes from Drive checkpoints, and a finished run is skipped automatically.

## Settings (the only cell you edit)

In [ ]:
BRANCH = "early-exit-part1"                    # git branch with the code
CONFIG = "configs/ee_mid_selfdistill.yaml"     # which run (Option B, mid-size)

RUN_TESTS = True       # unit + model tests (a few minutes); turn off once they pass
TRAIN = True           # Part 1 training; already-finished runs are skipped automatically
LAYER_REPORT = True    # Checkpoint 1: per-layer agreement / confidence report
EARLY_EXIT_EVAL = True # Part 2: translate with early stopping, speed/quality table + chart
NUM_SENTENCES = 300    # Part 2 eval sentences (use 50 for a quick check)

## 1. Get the latest code
Clones the repo, or updates an existing clone to the newest commit on `BRANCH`.
Note: this discards any edits made to the code files inside Colab.

In [ ]:
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/itsani-cmd/Distillation_cost_saver.git"
REPO_DIR = "/content/Distillation_cost_saver"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout {BRANCH}
        !git -C {REPO_DIR} reset --hard origin/{BRANCH}
    os.chdir(REPO_DIR)

print("cwd:", os.getcwd())
!git log --oneline -1

## 2. Install dependencies, mount Drive
Checkpoints, logs and results are saved to Google Drive, so a disconnect loses nothing.
If you re-run this notebook after the code changed, use **Runtime → Restart session** first.

In [ ]:
from src.config import load_config
from utils.colab_utils import get_project_paths, setup_colab

cfg = load_config(CONFIG)
paths = setup_colab(run_name=cfg.run_name)
print("run:", cfg.run_name, "| early exit:", cfg.early_exit.enabled,
      "| self_distill_weight:", cfg.early_exit.self_distill_weight)

## 3. Tests
The key checks: never stopping early reproduces normal greedy decoding exactly, and
"always stop at layer k" equals a model with only k decoder layers.
If anything fails here, stop and look at the error before training.

In [ ]:
if RUN_TESTS:
    !python -m pytest -q
    !RUN_HEAVY_TESTS=1 python -m pytest -q tests/test_early_exit_model.py
else:
    print("skipped (RUN_TESTS = False)")

## 4. Part 1 — train every decoder layer to guess
Log lines show `ce_L1 … ce_L6` (each layer's loss; should go down, L1 highest) and `sd`
(self-distillation). If Colab disconnects, re-run the notebook: training resumes from the
last finished epoch. A finished run returns immediately.

In [ ]:
if TRAIN:
    !python -m src.train --config {CONFIG}
else:
    print("skipped (TRAIN = False)")

## 5. Checkpoint 1 — did the early layers learn?
For each layer: accuracy, **agreement with the last layer** (key number), and confidence when it
agrees vs disagrees. `no_adapter` is the untrained model, for reference.

In [ ]:
if LAYER_REPORT:
    !python -m src.early_exit.analysis --config {CONFIG} --no-adapter
    !python -m src.early_exit.analysis --config {CONFIG}
else:
    print("skipped (LAYER_REPORT = False)")

## 6. Part 2 — translate with early stopping
Runs the full model (baseline), static exits at layers 1–5, the softmax-gap rule over a threshold
sweep, and the oracle (upper bound). All greedy, so compare rows with the `full` row.

In [ ]:
if EARLY_EXIT_EVAL:
    !python -m src.early_exit.run_eval --config {CONFIG} --num-sentences {NUM_SENTENCES}
else:
    print("skipped (EARLY_EXIT_EVAL = False)")

## 7. Results: table and chart

In [ ]:
import pandas as pd
from IPython.display import Image, display

out_dir = get_project_paths(cfg.run_name).outputs / "early_exit_eval"
csv_path = out_dir / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    cols = ["name", "bleu", "bleu_vs_full", "avg_layers", "layer_reduction"]
    display(df[cols].round(2))
    if (out_dir / "tradeoff.png").exists():
        display(Image(str(out_dir / "tradeoff.png")))
else:
    print("no results yet:", csv_path)

**What to look for**
- `full`: 6.00 layers, `bleu_vs_full` = 100 (the baseline).
- `static_L*`: BLEU should fall quickly as k gets smaller.
- `softmax_*`: find a threshold that cuts `avg_layers` noticeably while BLEU stays close to `full`;
  it should beat a static row with a similar number of layers.
- `oracle`: near-full BLEU with fewer layers (the best case).